In [ ]:
import matplotlib.pyplot as plt
import lightkurve as lk
import numpy as np
import pandas as pd
from astropy.timeseries import LombScargle

target_list = pd.read_csv("best_periods_unchecked.csv")

In [ ]:
idx = 1397
best_lc_number = int(target_list.loc[idx, "Best LC"]) - 1
kepler_id = int(target_list.loc[idx,"Kepler ID"])


search_result = lk.search_lightcurve(f"KIC {kepler_id}", author='Kepler')
best_lc = search_result[best_lc_number] 
lc = best_lc.download()
#lc = search_result.download()

time = lc.time.value if hasattr(lc.time, "value") else np.asarray(lc.time)
flux = lc.flux.value if hasattr(lc.flux, "value") else np.asarray(lc.flux)
    
mask = np.isfinite(time) & np.isfinite(flux)
time = time[mask]
flux = flux[mask]

frequency, power = LombScargle(time, flux).autopower(minimum_frequency=1/40, maximum_frequency=10)

peak_frequency = frequency[np.argmax(power)]
period = 1 / peak_frequency
peak_power = power[np.argmax(power)]

print(f"KIC {kepler_id}, Period: {period} days, Power: {peak_power}")

#Plot periodogram
plt.plot(1/frequency, power)
plt.xlabel('Period (days)')
plt.ylabel('Power')
plt.title('Lomb-Scargle Periodogram')
plt.xlim(0, 40)
plt.show()

#Plot folded LC
period = period/2
phase = (time % period) / period
sort_idx = np.argsort(phase)
phase = phase[sort_idx]
flux = flux[sort_idx]

plt.figure(figsize=(8,4))
plt.scatter(phase, flux, s=5, color='blue')
plt.xlabel("Phase")
plt.ylabel("Flux")
plt.title(f"KIC {kepler_id} Folded by {period} days")
plt.show()

#Plot LC without folding
lc.plot()